# Изградња апликација за генерисање слика (Azure OpenAI)

Велика језграна модела (LLM) нису само за генерисање текста. Такође можете генерисати слике из текстуалних описа. Слике као модалитет су корисне у МедТецху, архитектури, туризму, развоју игара, маркетингу и још много тога. У овој лекцији радимо са данашњим **GPT Image** моделима на Microsoft Foundry-у.

## Циљеви учења

До краја ове лекције моћи ћете да:

- Објасните шта је генерисање слика и где је корисно.
- Разумете породица модела `gpt-image` и како се разликује од старих DALL·E модела.
- Изградите апликацију за генерисање слика и уређујете слике.

## Шта је генерисање слика?

Модели за генерисање слика креирају слике из текстуалног упутства (промпта). Модерни модели као што је `gpt-image` уче како текст и слике међусобно функционишу током тренинга, а затим итеративно претварају случајни шум у слику која одговара вашем упутству.

Две добро познате породице модела слика су:

- **`gpt-image` (OpenAI)** — тренутна генерација која се користи у овој лекцији. Подржава генерисање слика из текста и уређивање слика (inpainting са маском).
- **Midjourney** — популаран модел треће стране са својом услугом и радним процесом заснованим на Discord-у.

> Старији OpenAI модели слика — **DALL·E 2** и **DALL·E 3** — су наследни. DALL·E 3 више није доступан за нове имплементације, а функције као што је `create_variation` постојале су само у DALL·E 2. За нове апликације користите `gpt-image` моделе.

На Microsoft Foundry-у, **`gpt-image-2`** је најновији и најспособнији модел слика и препоручени подразумевани избор. `gpt-image-1.5` и `gpt-image-1-mini` су такође доступни.

> **Важно:** `gpt-image` модели враћају генерисану слику као **base64** (`b64_json`), не као URL. Ваш код декодира base64 стринг у бајтове и чува је — не постоји URL слике за преузимање.


## Прављење ваше прве апликације за генерисање слика

Шта је потребно за прављење апликације за генерисање слика? Потребна вам је следећа библиотека:

- **python-dotenv**, препоручује се да користите ову библиотеку да бисте чували своје тајне у *.env* фајлу ван кода.
- **openai**, ову библиотеку ћете користити за интеракцију са OpenAI API-ем.
- **pillow**, за рад са сликама у Пајтону.

Ако то већ нисте урадили, пратите упутства на страници [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) да бисте креирали Microsoft Foundry ресурс и модел. Изаберите **gpt-image-2** као модел (најновији Azure OpenAI модел за слике; DALL·E је застарео).

1. Креирајте фајл *.env* са следећим садржајем:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Информације пронађите у Microsoft Foundry порталу за ваш ресурс у одељку "Deployments".



1. Сакупите горе наведене библиотеке у фајл који се зове *requirements.txt* на следећи начин:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Затим, направите виртуелно окружење и инсталирајте библиотеке:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Windows, користите следеће команде за креирање и активирање вашег виртуелног окружења:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Додајте следећи код у фајл који се зове *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # конфигуришите Azure OpenAI (Microsoft Foundry) клијент.
    # Модели слика захтевају новију верзију API-ја — проверите документацију Foundry за верзију која је потребна вашем моделу.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Креирајте слику користећи API за генерисање слика
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Унесите текст упита овде
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # нпр. "gpt-image-2"
        )
        # Поставите директоријум за сачувану слику
        image_dir = os.path.join(os.curdir, 'images')

        # Ако директоријум не постоји, креирајте га
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Иницијализујте путању до слике (наведите да је тип датотеке png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image модели враћају слику у base64 формату (b64_json), а не URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Прикажи слику у подразумеваном прегледачу слика
        image = Image.open(image_path)
        image.show()

    # ухвати изузетке
    except BadRequestError as err:
        print(err)

    ```

Хајде да објаснимо овај код:

- Прво, увозимо библиотеке које нам требају, укључујући OpenAI библиотеку, dotenv библиотеку, base64 модул и Pillow библиотеку.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Затим учитавамо променљиве окружења из *.env* фајла.

    ```python
    # увези dotenv
    dotenv.load_dotenv()
    ```

- Након тога, конфигуришемо Azure OpenAI (Microsoft Foundry) клијента.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Затим генеришемо слику:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Унесите ваш текст упита овде
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` модели враћају слику као **base64** стринг у `data[0].b64_json`. Декодујемо га у бајтове и записујемо у фајл — нема URL за преузимање.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- На крају, отварамо слику и користимо стандардни прегледач слика да је прикажемо:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Више детаља о генерисању слике

Погледајмо параметре `images.generate`:

- **prompt**, је текст који се користи за генерисање слике. Овде је то "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, је величина генерисане слике (на пример `1024x1024`, `1536x1024`, `1024x1536`, или `"auto"`).
- **n**, је број генерисаних слика. Овде генеришемо једну.
- **model**, је назив ваше сликовне моделе за распоређивање (на пример `gpt-image-2`).

> Сликовни модели не користе параметар `temperature` — то је контролa генерисања текста. Да бисте добили варијанте, позовите API поново; да бисте смањили варијанту, направите ваш prompt конкретнијим.

## Додатне могућности генерисања слике

Видели сте како се генерише слика са неколико редова Питон кода. `gpt-image` модели такође могу **уредити** постојећу слику. Обезбеђивањем слике, опционалне **маске** (која означава подручје за измену), и prompt-а, можете променити део слике — на пример, додати шешир нашем зеци.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# измене се такође враћају као base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основна слика садржи само зеца; коначна слика има шешир на зецу.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
